In [19]:
!pip install -q -U google-genai pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.6 MB/s eta 0:00:00


In [22]:
from google import genai
from google.genai import types
from google.colab import userdata
import numpy as np
import json
client=genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))
EMB_MODEL="gemini-embedding-001"
MODEL="gemini-3.5-flash-lite"
EMB_DIM=768

In [7]:
#1.Prepare Document
#1.1 Upload Document
from google.colab import files
uploaded=files.upload()
pdf_name=list(uploaded.keys())[0]
print(f"{pdf_name} is uploaded")

Saving College_FAQ_Knowledge_Base.pdf to College_FAQ_Knowledge_Base.pdf
College_FAQ_Knowledge_Base.pdf is uploaded


In [9]:
#1.2 Extract text
from pypdf import PdfReader
reader=PdfReader(pdf_name)
print("NUMBER OF PAGES: ",len(reader.pages))
text=""
for page in reader.pages:
  text+=page.extract_text()+"\n"
print("NUMBER OF CHARACTERS: ",len(text))

NUMBER OF PAGES:  3
NUMBER OF CHARACTERS:  5745


In [10]:
#1.3 Overalaping Chunking
def chunk_text(text,chunk_size=800,overlap=15):
  chunks=[]
  start=0
  while start<len(text):
    end=start+chunk_size
    chunks.append(text[start:end])
    start=end-overlap
  return chunks
chunks=chunk_text(text)
print("NUMBER OF CHUNKS: ",len(chunks))

NUMBER OF CHUNKS:  8


In [17]:
import numpy as np
#1.4 Embeddings
def embed(chunk):
  response=client.models.embed_content(
      model=EMB_MODEL,
      contents=chunk,
      config=types.EmbedContentConfig(
          output_dimensionality=EMB_DIM
      )
  )
  return response.embeddings[0].values

chunk_embed=[]
for chunk in chunks:
  chunk_embed.append(embed(chunk))
chunk_embed=np.array(chunk_embed,dtype="float32")
print("SHAPE: ",chunk_embed.shape)

SHAPE:  (8, 768)


In [27]:
import faiss
dimension=chunk_embed.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(chunk_embed)

In [32]:
system_instruction=f"""

Act as an AI Customer Support Assistant.

I will provide you with an issue that a user is facing.

Along with the issue, I will provide the top 3 most similar chunks retrieved from a knowledge-base PDF.
The PDF has already been processed by:
1. Extracting the text
2. Splitting the text into chunks
3. Generating embeddings
4. Performing similarity search

Your task is to answer the user's issue using ONLY the information available in the provided similar chunks.

Do not assume, invent, or add any information that is not available in the provided chunks.

If sufficient information to answer the issue is not available in the provided chunks, clearly state that the information is not available.

return text format where you take the relavant chunk then format answer and return formatted answer.no need relavent chunk only answer no heading
MOST IMPORTANT: On every starting of new chatbot session reply Hi,I am your Customer Support Assistant.
"""

chats=client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        temperature=0.2,
        max_output_tokens=2000,
        system_instruction=system_instruction
    ),
    history=[]
)
print("NOTE:ENTER quit bye exit to end session")
while True:
  user_input=input("Enter the query: ")
  if user_input.lower() in ["quit","bye","exit"]:
    break
  query_vector=np.array(embed(user_input),dtype="float32").reshape(1,-1)
  distances,indices=index.search(query_vector,3)
  sim_chunk=[chunks[i] for i in indices[0]]
  prompt=f"""
User Issue:
{user_input}

Retrieved Knowledge Base Chunks:
{sim_chunk}

Use ONLY the retrieved knowledge-base chunks above to answer the user's issue.

Do not use outside knowledge.
Do not assume missing information.
If the retrieved chunks do not contain enough information, state that the information is not available.
return text format where you take the relavant chunk then format answer and return formatted answer.
"""
  print(chats.send_message(prompt).text)



NOTE:ENTER quit bye exit to end session
Enter the query: tell me about attandance'
Hi,I am your Customer Support Assistant.

Here is the information regarding attendance:

* **Minimum Attendance Requirement:** Students are expected to maintain at least 85% attendance in each course unless a formally approved attendance relaxation applies.
* **Low Attendance Consequences:** Students falling below the required attendance percentage may be restricted from appearing for the relevant examination according to institutional academic rules.
* **Attendance Shortage Condonation:** Attendance shortage may be considered for condonation only when the student meets the applicable institutional conditions and obtains the required approval.
Enter the query: bye


In [5]:
for m in client.models.list():
  print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/